In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Task 2: Schema and Integrity

Validate the dataset schema and integrity, generate the column profile, and record the results of each check.

Final outputs:
- `profile.csv`
- `validation_results.csv`
- `anomaly_register.csv`

## Setup

In [2]:
def find_repo_root():
    path = Path.cwd().resolve()

    for parent in [path, *path.parents]:
        if (parent / "data" / "allstate_claims_data.csv").exists():
            return parent

    raise FileNotFoundError("Could not find data/allstate_claims_data.csv")


REPO_ROOT = find_repo_root()

DATA_PATH = REPO_ROOT / "data" / "allstate_claims_data.csv"

OUTPUT_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 1"
)

PROFILE_PATH = OUTPUT_DIR / "profile.csv"
VALIDATION_PATH = OUTPUT_DIR / "validation_results.csv"
ANOMALY_PATH = OUTPUT_DIR / "anomaly_register.csv"

ID_COL = "id"
CAT_COLS = [f"cat{i}" for i in range(1, 117)]
CONT_COLS = [f"cont{i}" for i in range(1, 15)]
TARGET_COL = "loss"

EXPECTED_ROWS = 188_318
EXPECTED_COLS = 132
EXPECTED_SIZE = 70_025_339

EXPECTED_HEADER = (
    [ID_COL]
    + CAT_COLS
    + CONT_COLS
    + [TARGET_COL]
)

print(f"Source: {DATA_PATH.relative_to(REPO_ROOT)}")

Source: data\allstate_claims_data.csv


## Load Data

Load the authoritative source and confirm its basic identity before continuing.

In [3]:
raw_df = pd.read_csv(DATA_PATH)

file_size = DATA_PATH.stat().st_size
shape = raw_df.shape
header_match = raw_df.columns.tolist() == EXPECTED_HEADER

print(f"File size: {file_size:,} bytes")
print(f"Shape: {shape}")
print(f"Header match: {header_match}")

File size: 70,025,339 bytes
Shape: (188318, 132)
Header match: True


In [4]:
source_checks = {
    "file size": file_size == EXPECTED_SIZE,
    "row count": raw_df.shape[0] == EXPECTED_ROWS,
    "column count": raw_df.shape[1] == EXPECTED_COLS,
    "header": header_match,
}

for check, passed in source_checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

if not all(source_checks.values()):
    raise RuntimeError(
        "Source identity check failed. "
        "Run Task 1 and review the source before continuing."
    )

file size: PASS
row count: PASS
column count: PASS
header: PASS


## Column Roles

Expected roles:
- 1 identifier
- 116 categorical predictors
- 14 continuous predictors
- 1 regression target

In [5]:
role_results = pd.DataFrame({
    "role": [
        "identifier",
        "categorical predictors",
        "continuous predictors",
        "regression target",
    ],
    "observed": [
        1,
        len(CAT_COLS),
        len(CONT_COLS),
        1,
    ],
    "expected": [
        1,
        116,
        14,
        1,
    ],
})

role_results

,role,observed,expected
0,identifier,1,1
1,categorical predictors,116,116
2,continuous predictors,14,14
3,regression target,1,1


## Categorical Fields

Treat all `cat*` fields as unordered category labels.

In [6]:
df = raw_df.copy()

for col in CAT_COLS:
    df[col] = pd.Categorical(
        df[col].astype("string"),
        ordered=False
    )

print(f"Converted {len(CAT_COLS)} categorical fields.")

Converted 116 categorical fields.


## ID Check

`id` should be present, non-missing, and unique across all 188,318 rows.

In [7]:
id_present = ID_COL in df.columns
id_missing = int(df[ID_COL].isna().sum())
id_unique = int(df[ID_COL].nunique(dropna=False))
id_duplicates = int(df[ID_COL].duplicated().sum())

print(f"Present: {id_present}")
print(f"Missing: {id_missing}")
print(f"Unique: {id_unique:,}")
print(f"Duplicates: {id_duplicates}")

Present: True
Missing: 0
Unique: 188,318
Duplicates: 0


## Missingness and Duplicates

The table should contain no exact duplicate rows and no missing cells.

In [8]:
duplicate_rows = int(raw_df.duplicated().sum())
missing_cells = int(raw_df.isna().sum().sum())

print(f"Exact duplicate rows: {duplicate_rows}")
print(f"Missing cells: {missing_cells}")

Exact duplicate rows: 0
Missing cells: 0


## Continuous Fields

All `cont*` fields should be numeric, finite, and within the documented 0-to-1 range.

In [9]:
continuous_checks = []

for col in CONT_COLS:
    original = raw_df[col]
    numeric = pd.to_numeric(original, errors="coerce")

    parse_failures = int(
        (numeric.isna() & original.notna()).sum()
    )

    non_finite = int(
        (~np.isfinite(numeric.dropna())).sum()
    )

    below_zero = int((numeric < 0).sum())
    above_one = int((numeric > 1).sum())

    continuous_checks.append({
        "column": col,
        "dtype": str(original.dtype),
        "numeric": (
            pd.api.types.is_numeric_dtype(original)
            and parse_failures == 0
        ),
        "min": numeric.min(),
        "max": numeric.max(),
        "parse_failures": parse_failures,
        "non_finite": non_finite,
        "below_0": below_zero,
        "above_1": above_one,
        "range_violations": below_zero + above_one,
    })

continuous_results = pd.DataFrame(continuous_checks)

continuous_results

,column,dtype,numeric,min,max,parse_failures,non_finite,below_0,above_1,range_violations
0,cont1,float64,True,0.000016,0.984975,0,0,0,0,0
1,cont2,float64,True,0.001149,0.862654,0,0,0,0,0
2,cont3,float64,True,0.002634,0.944251,0,0,0,0,0
3,cont4,float64,True,0.176921,0.954297,0,0,0,0,0
4,cont5,float64,True,0.281143,0.983674,0,0,0,0,0
5,cont6,float64,True,0.012683,0.997162,0,0,0,0,0
6,cont7,float64,True,0.069503,1.000000,0,0,0,0,0
7,cont8,float64,True,0.236880,0.980200,0,0,0,0,0
8,cont9,float64,True,0.000080,0.995400,0,0,0,0,0
9,cont10,float64,True,0.000000,0.994980,0,0,0,0,0


In [10]:
all_cont_numeric = bool(
    continuous_results["numeric"].all()
)

cont_parse_failures = int(
    continuous_results["parse_failures"].sum()
)

cont_nonfinite = int(
    continuous_results["non_finite"].sum()
)

cont_range_violations = int(
    continuous_results["range_violations"].sum()
)

print(f"All numeric: {all_cont_numeric}")
print(f"Parse failures: {cont_parse_failures}")
print(f"Non-finite values: {cont_nonfinite}")
print(f"Range violations: {cont_range_violations}")

All numeric: True
Parse failures: 0
Non-finite values: 0
Range violations: 0


## Loss

`loss` should be numeric, finite, and strictly greater than 0.

Invalid values are recorded, not removed.

In [11]:
loss_original = raw_df[TARGET_COL]

loss_values = pd.to_numeric(
    loss_original,
    errors="coerce"
)

loss_parse_failures = int(
    (loss_values.isna() & loss_original.notna()).sum()
)

loss_numeric = (
    pd.api.types.is_numeric_dtype(loss_original)
    and loss_parse_failures == 0
)

loss_nonfinite = int(
    (~np.isfinite(loss_values.dropna())).sum()
)

loss_nonpositive = int(
    (loss_values <= 0).sum()
)

print(f"Numeric: {loss_numeric}")
print(f"Parse failures: {loss_parse_failures}")
print(f"Non-finite values: {loss_nonfinite}")
print(f"Non-positive values: {loss_nonpositive}")

Numeric: True
Parse failures: 0
Non-finite values: 0
Non-positive values: 0


In [12]:
loss_violations = raw_df.loc[
    loss_values.isna()
    | (~np.isfinite(loss_values))
    | (loss_values <= 0)
]

print(f"Loss violations: {len(loss_violations)}")

loss_violations

Loss violations: 0


,id,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,...,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,loss


## Categorical Validation

Confirm that every `cat*` field is stored as an unordered category.

In [13]:
categorical_checks = []

for col in CAT_COLS:
    categorical_checks.append({
        "column": col,
        "source_dtype": str(raw_df[col].dtype),
        "workflow_dtype": str(df[col].dtype),
        "is_category": isinstance(
            df[col].dtype,
            pd.CategoricalDtype
        ),
        "ordered": bool(df[col].cat.ordered),
        "n_unique": int(df[col].nunique(dropna=False)),
    })

categorical_results = pd.DataFrame(categorical_checks)

categorical_results

,column,source_dtype,workflow_dtype,is_category,ordered,n_unique
0,cat1,object,category,True,False,2
1,cat2,object,category,True,False,2
2,cat3,object,category,True,False,2
3,cat4,object,category,True,False,2
4,cat5,object,category,True,False,2
...,...,...,...,...,...,...
111,cat112,object,category,True,False,51
112,cat113,object,category,True,False,61
113,cat114,object,category,True,False,19
114,cat115,object,category,True,False,23


In [14]:
all_categories = bool(
    categorical_results["is_category"].all()
)

all_unordered = bool(
    (~categorical_results["ordered"]).all()
)

print(f"All categorical: {all_categories}")
print(f"All unordered: {all_unordered}")

All categorical: True
All unordered: True


## Column Profile

Create the machine-readable profile for all 132 fields.

The profile is built in memory first and is only saved after every validation check passes.

In [15]:
profile_rows = []

for col in df.columns:
    if col == ID_COL:
        role = "identifier"
    elif col in CAT_COLS:
        role = "categorical_predictor"
    elif col in CONT_COLS:
        role = "continuous_predictor"
    elif col == TARGET_COL:
        role = "regression_target"
    else:
        role = "unexpected"

    source_series = raw_df[col]
    workflow_series = df[col]

    is_numeric = pd.api.types.is_numeric_dtype(source_series)

    profile_rows.append({
        "column": col,
        "role": role,
        "source_dtype": str(source_series.dtype),
        "workflow_dtype": str(workflow_series.dtype),
        "missing": int(source_series.isna().sum()),
        "n_unique": int(source_series.nunique(dropna=False)),
        "min": source_series.min() if is_numeric else None,
        "max": source_series.max() if is_numeric else None,
    })

profile = pd.DataFrame(profile_rows)

print(f"Profile rows: {len(profile)}")

profile

Profile rows: 132


,column,role,source_dtype,workflow_dtype,missing,n_unique,min,max
0,id,identifier,int64,int64,0,188318,1.000000,587633.000000
1,cat1,categorical_predictor,object,category,0,2,NaN,NaN
2,cat2,categorical_predictor,object,category,0,2,NaN,NaN
3,cat3,categorical_predictor,object,category,0,2,NaN,NaN
4,cat4,categorical_predictor,object,category,0,2,NaN,NaN
...,...,...,...,...,...,...,...,...
127,cont11,continuous_predictor,float64,float64,0,326,0.035321,0.998742
128,cont12,continuous_predictor,float64,float64,0,328,0.036232,0.998484
129,cont13,continuous_predictor,float64,float64,0,353,0.000228,0.988494
130,cont14,continuous_predictor,float64,float64,0,18740,0.179722,0.844848


## Validation Results

Collect all required checks into one machine-readable table.

In [16]:
validation_results = pd.DataFrame([
    {
        "check": "row count",
        "observed": raw_df.shape[0],
        "expected": EXPECTED_ROWS,
        "status": "pass" if raw_df.shape[0] == EXPECTED_ROWS else "fail",
    },
    {
        "check": "column count",
        "observed": raw_df.shape[1],
        "expected": EXPECTED_COLS,
        "status": "pass" if raw_df.shape[1] == EXPECTED_COLS else "fail",
    },
    {
        "check": "header order",
        "observed": header_match,
        "expected": True,
        "status": "pass" if header_match else "fail",
    },
    {
        "check": "identifier count",
        "observed": 1,
        "expected": 1,
        "status": "pass",
    },
    {
        "check": "categorical predictor count",
        "observed": len(CAT_COLS),
        "expected": 116,
        "status": "pass" if len(CAT_COLS) == 116 else "fail",
    },
    {
        "check": "continuous predictor count",
        "observed": len(CONT_COLS),
        "expected": 14,
        "status": "pass" if len(CONT_COLS) == 14 else "fail",
    },
    {
        "check": "target count",
        "observed": 1,
        "expected": 1,
        "status": "pass",
    },
    {
        "check": "id present",
        "observed": id_present,
        "expected": True,
        "status": "pass" if id_present else "fail",
    },
    {
        "check": "missing ids",
        "observed": id_missing,
        "expected": 0,
        "status": "pass" if id_missing == 0 else "fail",
    },
    {
        "check": "unique ids",
        "observed": id_unique,
        "expected": EXPECTED_ROWS,
        "status": "pass" if id_unique == EXPECTED_ROWS else "fail",
    },
    {
        "check": "duplicate ids",
        "observed": id_duplicates,
        "expected": 0,
        "status": "pass" if id_duplicates == 0 else "fail",
    },
    {
        "check": "duplicate rows",
        "observed": duplicate_rows,
        "expected": 0,
        "status": "pass" if duplicate_rows == 0 else "fail",
    },
    {
        "check": "missing cells",
        "observed": missing_cells,
        "expected": 0,
        "status": "pass" if missing_cells == 0 else "fail",
    },
    {
        "check": "continuous fields numeric",
        "observed": all_cont_numeric,
        "expected": True,
        "status": "pass" if all_cont_numeric else "fail",
    },
    {
        "check": "continuous parse failures",
        "observed": cont_parse_failures,
        "expected": 0,
        "status": "pass" if cont_parse_failures == 0 else "fail",
    },
    {
        "check": "non-finite continuous values",
        "observed": cont_nonfinite,
        "expected": 0,
        "status": "pass" if cont_nonfinite == 0 else "fail",
    },
    {
        "check": "continuous range violations",
        "observed": cont_range_violations,
        "expected": 0,
        "status": "pass" if cont_range_violations == 0 else "fail",
    },
    {
        "check": "loss numeric",
        "observed": loss_numeric,
        "expected": True,
        "status": "pass" if loss_numeric else "fail",
    },
    {
        "check": "loss parse failures",
        "observed": loss_parse_failures,
        "expected": 0,
        "status": "pass" if loss_parse_failures == 0 else "fail",
    },
    {
        "check": "non-finite loss values",
        "observed": loss_nonfinite,
        "expected": 0,
        "status": "pass" if loss_nonfinite == 0 else "fail",
    },
    {
        "check": "non-positive loss values",
        "observed": loss_nonpositive,
        "expected": 0,
        "status": "pass" if loss_nonpositive == 0 else "fail",
    },
    {
        "check": "categorical fields",
        "observed": all_categories,
        "expected": True,
        "status": "pass" if all_categories else "fail",
    },
    {
        "check": "categorical fields unordered",
        "observed": all_unordered,
        "expected": True,
        "status": "pass" if all_unordered else "fail",
    },
    {
        "check": "profile rows",
        "observed": len(profile),
        "expected": EXPECTED_COLS,
        "status": "pass" if len(profile) == EXPECTED_COLS else "fail",
    },
])

validation_results

,check,observed,expected,status
0,row count,188318,188318,pass
1,column count,132,132,pass
2,header order,True,True,pass
3,identifier count,1,1,pass
4,categorical predictor count,116,116,pass
5,continuous predictor count,14,14,pass
6,target count,1,1,pass
7,id present,True,True,pass
8,missing ids,0,0,pass
9,unique ids,188318,188318,pass


In [17]:
saved_profile = pd.read_csv(PROFILE_PATH)
saved_validation = pd.read_csv(VALIDATION_PATH)

assert saved_profile["column"].tolist() == EXPECTED_HEADER
assert len(saved_profile) == EXPECTED_COLS

assert len(saved_validation) == len(validation_results)
assert saved_validation["status"].eq("pass").all()

print("Published Gate 1 artifacts: PASS")

Published Gate 1 artifacts: PASS


In [18]:
failed_checks = validation_results.loc[
    validation_results["status"] == "fail"
].copy()

candidate_anomalies = pd.DataFrame({
    "check": failed_checks["check"],
    "observed": failed_checks["observed"],
    "expected": failed_checks["expected"],
    "severity": "pending review",
    "status": "detected",
    "evidence": "task_2.ipynb validation results",
})

candidate_anomalies

,check,observed,expected,severity,status,evidence


## Anomaly Check

Failed validations are shown here for review.

Do not automatically add them to the official anomaly register until the team confirms they are real source, parsing, schema, or integrity anomalies.

In [19]:
failed_checks = validation_results[
    validation_results["status"] == "fail"
].copy()

failed_checks

,check,observed,expected,status


## Anomaly Register

The official anomaly register contains confirmed anomalies only.

In [20]:
ANOMALY_COLUMNS = [
    "id",
    "description",
    "affected_count",
    "severity",
    "owner",
    "status",
    "evidence",
    "handling_decision",
    "likely_impact",
]

if ANOMALY_PATH.exists():
    anomaly_register = pd.read_csv(ANOMALY_PATH)
else:
    anomaly_register = pd.DataFrame(
        columns=ANOMALY_COLUMNS
    )

anomaly_register

,id,description,affected_count,severity,owner,status,evidence,handling_decision,likely_impact


## Workflow Result

A clean run requires every validation check to pass.

In [21]:
passed_count = int(
    (validation_results["status"] == "pass").sum()
)

failed_count = int(
    (validation_results["status"] == "fail").sum()
)

print(f"Checks passed: {passed_count}")
print(f"Checks failed: {failed_count}")

if failed_count == 0:
    print("\nCLEAN WORKFLOW RUN: PASS")
else:
    print("\nCLEAN WORKFLOW RUN: FAIL")
    print("Review the failed checks before continuing.")
    display(failed_checks)

Checks passed: 24
Checks failed: 0

CLEAN WORKFLOW RUN: PASS


In [22]:
if failed_count > 0:
    raise RuntimeError(
        "Validation failed. Final CSV outputs were not changed."
    )

## Save Outputs

Only a clean workflow run updates the final profile and validation results.

In [23]:
profile.to_csv(
    PROFILE_PATH,
    index=False
)

validation_results.to_csv(
    VALIDATION_PATH,
    index=False
)

# Create the anomaly register if it does not exist yet.
if not ANOMALY_PATH.exists():
    pd.DataFrame(
        columns=ANOMALY_COLUMNS
    ).to_csv(
        ANOMALY_PATH,
        index=False
    )

print("Updated final outputs:")
print(f"- {PROFILE_PATH.name}")
print(f"- {VALIDATION_PATH.name}")
print(f"- {ANOMALY_PATH.name}")

Updated final outputs:
- profile.csv
- validation_results.csv
- anomaly_register.csv
